# Weak-class audit of epoch96 best weights
Attach BTP-code-weak-audit.zip, quarter_lr_to_epoch100_20260918_094320.zip, and prepared public252 data. Automatic extraction. One complete validation evaluation, no training or test evaluation. Save the final ZIP. Preview colors are the fixed dataset palette; pseudo-RGB is display-scaled only.

In [ ]:
from pathlib import Path
import shutil, zipfile, subprocess, sys
INPUTS = Path('/kaggle/input')
CODE = Path('/kaggle/working/BTP-weak-audit')
def safe_extract(z, dest):
    dest = dest.resolve()
    for member in z.infolist():
        if not (dest / member.filename).resolve().is_relative_to(dest):
            raise ValueError('Unsafe ZIP member')
    z.extractall(dest)
if not CODE.exists():
    sources = list(INPUTS.rglob('paired_full_training.py'))
    if len(sources) == 1:
        shutil.copytree(sources[0].parent, CODE)
    else:
        found = []
        for archive in INPUTS.rglob('*.zip'):
            if zipfile.is_zipfile(archive):
                with zipfile.ZipFile(archive) as z:
                    if 'BTP/paired_full_training.py' in z.namelist(): found.append(archive)
        assert len(found) == 1, 'Attach exactly one BTP-code-weak-audit.zip input.'
        staging = Path('/kaggle/working/weak_audit_code_extract')
        staging.mkdir(exist_ok=False)
        with zipfile.ZipFile(found[0]) as z: safe_extract(z, staging)
        shutil.copytree(staging / 'BTP', CODE)
assert (CODE / 'paired_full_training.py').is_file()
import json, hashlib, math
import torch
expected = 'quarter_lr_to_epoch100_20260918_094320'
relative = expected + '/model/last.pt'
refs = list(INPUTS.rglob(relative))
if not refs:
    working = Path('/kaggle/working') / relative
    if working.is_file(): refs = [working]
if not refs:
    dest = Path('/kaggle/working/restored_quarter60')
    restored = dest / relative
    if restored.is_file(): refs = [restored]
    else:
        for archive in INPUTS.rglob('*.zip'):
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                if relative in z.namelist():
                    safe_extract(z, dest)
                    refs = [dest / relative]
                    break
assert len(refs) == 1, 'Attach quarter_lr_to_epoch100_20260918_094320.zip or its extracted folder.'
CHECKPOINT = refs[0]
for name in ['last.pt', 'best_iou.pth', 'best_psnr.pth']:
    assert CHECKPOINT.with_name(name).is_file(), f'Missing {name}'
CHECKPOINT = CHECKPOINT.with_name('best_iou.pth')
assert '--save_segmentation' in (CODE / 'test.py').read_text(), 'Attach the updated BTP-code-weak-audit.zip.'
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert torch.cuda.is_available(), 'Enable the GPU.'
print('Evaluating saved best segmentation checkpoint:', CHECKPOINT)


In [ ]:
from datetime import datetime
from IPython.display import display, FileLink
NAME = 'weak_class_audit_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = Path('/kaggle/working') / NAME
OUT.mkdir(exist_ok=False)
try:
    subprocess.run([sys.executable, '-u', 'test.py', '--data_root', str(DATA),
        '--transpose_image', '--eval_split', 'val', '--save_segmentation',
        '--pretrained_model_path', str(CHECKPOINT), '--outf', str(OUT), '--name', 'best_epoch96'], cwd=CODE, check=True)
finally:
    archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=NAME)
    with zipfile.ZipFile(archive) as z: assert z.testzip() is None
    import os
    os.chdir('/kaggle/working')
    print('Download:', archive)
    display(FileLink(Path(archive).name))


In [ ]:
import numpy as np
import sys
sys.path.insert(0,str(CODE))
from dataset import get_class_names
folder = OUT / 'best_epoch96/evaluation_val'
meta = json.loads((folder / 'evaluation.json').read_text())
assert meta['status']=='completed' and meta['scene_count']==25
hist = np.load(folder / 'confusion_matrix.npy')
classes = get_class_names()
for k in [9,12,17,22]:
    total = hist[k].sum()
    print(classes[k], 'ground-truth pixels:', int(total))
    print([(classes[j], round(100*hist[k,j]/total,2)) for j in np.argsort(hist[k])[::-1][:5]])
print('Share the downloaded ZIP; it contains all25 previews and per-scene confusion matrices, without reconstructed HSI cubes.')
